# Riproduzione completa su Google Colab — *You Don't Bring Me Flowers* (RecSys 2025, Best Full Paper)

Notebook per il corso di Information Retrieval (prof. Narducci). Esegue il **protocollo sperimentale pubblicato dagli autori**
(repo `geektoni/mitigating-harm-recsys`, commit fissato) con **lo stesso codice e lo stesso dataset**: 6 modelli, 100 epoche, 10 run.

**Struttura**

| Parte | Cosa fa | Tempo indicativo |
|---|---|---|
| **A** | Setup: Drive, clone del repo al commit fissato, dipendenze, controlli | ~10 min |
| **B** | Rigenera le figure del paper dai CSV *pubblicati* dagli autori (verifica dell'analisi) | pochi minuti |
| **C** | Rigenera gli split e **riesegue la valutazione conformal** (script `experiments/*.sh`) sui punteggi pre-calcolati degli autori; confronto numerico con i loro CSV | dipende dalla CPU Colab (misuralo) |
| **D** | **Riaddestra da zero** i modelli (`train_rec.py`, 100 epoche × 10 run), con ripresa automatica su Drive; poi valutazione e confronto | molte ore: va spezzato su più sessioni |

Tutto ciò che conta (pickle riaddestrati, CSV, figure, log) viene salvato su **Google Drive**, così una disconnessione di Colab non fa perdere il lavoro:
basta rieseguire le celle e il notebook riprende da dove si era fermato.

**Scarto dal protocollo — elenco completo e onesto**

1. `usetex` disattivato nei 3 script di plotting (Colab non ha LaTeX): cambia solo il font delle figure, non i numeri.
2. `pandas<3` (il codice degli autori usa `include_groups=True`, rifiutato da pandas 3.x).
3. Installata la dipendenza **`torchsparsegradutils`**, che il loro `requirement.txt` *non elenca* ma `SIGformer/model.py` importa (senza, `train_rec.py` non parte nemmeno).
4. Variabile d'ambiente `PYTHONPATH=.` anche per i plot (il README la prevede solo per le valutazioni; senza, `plot_figure_5.py` e `plot_figure_6.py` falliscono con `No module named 'analytics'`).
5. `--gpu 0` invece di `--gpu 1` (Colab ha una sola GPU).
6. **Solo in Parte D e solo se `PER_RUN_RESUME=True`:** una patch di 2 righe a `train_rec.py` (`--only-run N`) per poter riprendere il training run per run dopo una disconnessione.
   Effetto collaterale dichiarato: gli autori lanciano le 10 run in un unico processo con seed fissato *una sola volta* (2025), quindi i pesi iniziali della run *k* dipendono dalle run precedenti;
   con la ripresa per-run ogni run riparte dal seed 2025. Gli split (`random_state=run_id`) restano identici. Non è comunque possibile la riproducibilità bit-a-bit su GPU.
   Se `PER_RUN_RESUME=False` il notebook lancia **esattamente** il comando degli autori (`--runs 10`), a costo di dover completare un modello in una sola sessione.

## Parte A — Setup

In [9]:
# ===== CONFIGURAZIONE (modifica solo se sai perche') =====
PAPER_COMMIT   = "f4dfa8a727d560cd6c7c0d99a981bff690ee6b7a"   # commit del repo ispezionato e usato per questo notebook
REPO_URL       = "https://github.com/geektoni/mitigating-harm-recsys.git"
DRIVE_DIR      = "/content/drive/MyDrive/IR_RecSys2025_repro"  # tutto l'output persistente finisce qui

EPOCHS, RUNS   = 100, 10        # protocollo ufficiale
GPU_INDEX      = 0              # Colab: una sola GPU
CORES          = 8              # come negli script ufficiali; se la RAM Colab esplode in Parte C/D prova 2 (non cambia i risultati, solo il parallelismo)

# Batch size: il README e methods/run_all.sh NON coincidono (vedi nota in Parte D). Default = run_all.sh.
BATCH_PROFILE  = "run_all"      # "run_all" | "readme"
INCLUDE_NCFHARM = True          # ncfharm e' in run_all.sh ma NON e' usato da experiments/*.sh

PER_RUN_RESUME = True           # vedi punto 6 dell'elenco scarti; False = comando degli autori esatto (--runs 10)
KEEP_INTERMEDIATE_CKPT = False  # i pickle a epoca 10,20,...,90 non servono alla valutazione (gli autori rilasciano solo l'epoca 100)
SESSION_BUDGET_HOURS = 10.5     # Parte D smette di lanciare nuove run oltre questo tempo (Colab free ~12h max)
EXTRA_PYG_EXTENSIONS = False    # torch_scatter/sparse/cluster/pyg_lib: il codice degli autori NON li importa direttamente; metti True solo se compare un ImportError

In [10]:
import os, sys, subprocess, shutil, time, re, glob, json, textwrap
from pathlib import Path

def sh(cmd, check=True, tail=2500, cwd=None, env=None):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd=cwd, env=env)
    out = p.stdout or ""
    print(out if len(out) <= tail else "...\n" + out[-tail:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Comando fallito (exit {p.returncode}): {cmd}")
    return out

print("Python:", sys.version.split()[0])
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader", check=False)
sh("nvidia-smi | head -4 | tail -2", check=False)
sh("nproc; free -g | head -2; df -h /content | tail -1", check=False)

Python: 3.13.16
$ nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
/bin/sh: 1: nvidia-smi: not found

$ nvidia-smi | head -4 | tail -2
/bin/sh: 1: nvidia-smi: not found

$ nproc; free -g | head -2; df -h /content | tail -1
2
               total        used        free      shared  buff/cache   available
Mem:              12           1           8           0           3          11
overlay         108G   22G   87G  21% /



'2\n               total        used        free      shared  buff/cache   available\nMem:              12           1           8           0           3          11\noverlay         108G   22G   87G  21% /\n'

In [11]:
from google.colab import drive
drive.mount("/content/drive")

D = Path(DRIVE_DIR)
DIRS = {k: D / k for k in [
    "retrain_pickles",        # pickle riaddestrati (equivalente di methods/kuairand/results)
    "eval_on_authors",        # CSV rigenerati da valutazione su pickle degli autori (Parte C)
    "eval_on_retrained",      # CSV da valutazione su pickle riaddestrati (Parte D)
    "figures_authors_csv",    # figure dai CSV degli autori (Parte B)
    "figures_eval_on_authors",
    "figures_retrained",
    "logs", "patches"]}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)
print("Cartella di lavoro su Drive:", D)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cartella di lavoro su Drive: /content/drive/MyDrive/IR_RecSys2025_repro


In [12]:
REPO_AUTHORS = Path("/content/repo_authors")     # copia INTATTA del repo (con i pickle pre-calcolati degli autori)
REPO_RETRAIN = Path("/content/repo_retrain")     # copia per il riaddestramento (Parte D)

if not REPO_AUTHORS.exists():
    sh(f"git clone -q {REPO_URL} {REPO_AUTHORS}")
sh(f"git -C {REPO_AUTHORS} checkout -q {PAPER_COMMIT} && git -C {REPO_AUTHORS} log -1 --format='%H %ci %s'")
sh(f"ls {REPO_AUTHORS}/KuaiRand-Harm/training/ && du -sh {REPO_AUTHORS}/methods/kuairand/results {REPO_AUTHORS}/results")
n_pk = len(glob.glob(str(REPO_AUTHORS / "methods/kuairand/results/*.zlib.pickle")))
print("Pickle pre-calcolati degli autori trovati:", n_pk, "(attesi 60 = 6 modelli x 10 run)")
assert n_pk == 60, "Il repo non contiene i 60 pickle attesi: controlla il commit"

$ git -C /content/repo_authors checkout -q f4dfa8a727d560cd6c7c0d99a981bff690ee6b7a && git -C /content/repo_authors log -1 --format='%H %ci %s'
f4dfa8a727d560cd6c7c0d99a981bff690ee6b7a 2026-06-11 16:28:13 +0200 Create LICENSE

$ ls /content/repo_authors/KuaiRand-Harm/training/ && du -sh /content/repo_authors/methods/kuairand/results /content/repo_authors/results
single_and_repeated_interactions_is_click.csv.gzip
464M	/content/repo_authors/methods/kuairand/results
904K	/content/repo_authors/results

Pickle pre-calcolati degli autori trovati: 60 (attesi 60 = 6 modelli x 10 run)


In [1]:
# ---- Dipendenze ----
# Il requirements.txt degli autori pinna torch 2.5.0+cu124 (wheel esistono solo fino a Python 3.12).
PINNED_TORCH = sys.version_info < (3, 13)
if PINNED_TORCH:
    sh("pip install -q torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu124", tail=1500)
else:
    print("Python >= 3.13: wheel torch 2.5.0 non disponibili -> uso il torch preinstallato di Colab (scarto dichiarato).")

sh('pip install -q "pandas>=2.2,<3" scipy scikit-learn matplotlib seaborn tqdm networkx torch_geometric', tail=1500)

# Dipendenza NON elencata nel requirement.txt degli autori ma importata da methods/SIGformer/model.py
sh("pip install -q torchsparsegradutils", check=False, tail=800)
if subprocess.run([sys.executable, "-c", "import torchsparsegradutils"], capture_output=True).returncode != 0:
    print("pip non ha trovato torchsparsegradutils: provo dal repository GitHub")
    sh("pip install -q git+https://github.com/cai4cai/torchsparsegradutils.git", tail=1500)

if EXTRA_PYG_EXTENSIONS:
    ver = subprocess.check_output([sys.executable, "-c", "import torch;print(torch.__version__.split('+')[0], 'cu'+torch.version.cuda.replace('.',''))"], text=True).split()
    sh(f"pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-{ver[0]}+{ver[1]}.html", tail=1500)

Path("/content/_check_env.py").write_text("""
import torch, pandas, numpy, sklearn, scipy, torch_geometric, torchsparsegradutils
print('torch', torch.__version__, '| cuda build', torch.version.cuda, '| cuda disponibile:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NESSUNA')
print('pandas', pandas.__version__, '| numpy', numpy.__version__, '| sklearn', sklearn.__version__, '| torch_geometric', torch_geometric.__version__)
assert int(pandas.__version__.split('.')[0]) < 3, 'pandas>=3 non e compatibile con gli script degli autori'
assert torch.cuda.is_available(), 'Attiva la GPU: Runtime > Cambia tipo di runtime > T4 GPU'
# smoke test dei modelli: gli stessi import di methods/train_rec.py
import sys; sys.path.insert(0, '/content/repo_authors/methods')
from LightGCL.model import LightGCL
from SIGformer.model import SIGformer
from GFormer.model import GFormer, GTLayer
from SiReN.model import SiReN
from NCF.model import RankerNN
print('Import di tutti i modelli: OK')
""")
sh(f"{sys.executable} /content/_check_env.py")

NameError: name 'sys' is not defined

In [ ]:
# ---- Patch documentate (ognuna con assert: se il repo cambia, il notebook si ferma invece di patchare alla cieca) ----
def patch_file(path, old, new, count=1):
    s = Path(path).read_text()
    assert s.count(old) == count, f"Patch non applicabile a {path}: attese {count} occorrenze, trovate {s.count(old)}: {old[:60]!r}"
    Path(path).write_text(s.replace(old, new))

# Patch 1 (copia 'authors'): usetex -> False nei 3 script di plotting (Colab non ha LaTeX)
for f in ["plot_figure_3_and_4.py", "plot_figure_5.py", "plot_figure_6.py"]:
    p = REPO_AUTHORS / "analytics" / f
    if '"text.usetex": True' in p.read_text():
        patch_file(p, '"text.usetex": True', '"text.usetex": False')
print("OK: usetex disattivato in analytics/*.py (copia authors)")

# Copia per il riaddestramento: stessi file, ma SENZA i pickle pre-calcolati (altrimenti il training li sovrascriverebbe)
if not REPO_RETRAIN.exists():
    shutil.copytree(REPO_AUTHORS, REPO_RETRAIN, symlinks=True, ignore=shutil.ignore_patterns(".git"))
res_dir = REPO_RETRAIN / "methods/kuairand/results"
if res_dir.is_symlink():
    pass
else:
    shutil.rmtree(res_dir)
    os.symlink(DIRS["retrain_pickles"], res_dir)     # i pickle riaddestrati vivono su Drive
print("methods/kuairand/results (copia retrain) ->", os.readlink(res_dir))

# Patch 2 (solo copia 'retrain'): --only-run N
tr = REPO_RETRAIN / "methods/train_rec.py"
if "--only-run" not in tr.read_text():
    orig = tr.read_text()
    patch_file(tr,
        'parser.add_argument("--filter-hate", default=False, action="store_true", help="Filter the data by hate")',
        'parser.add_argument("--filter-hate", default=False, action="store_true", help="Filter the data by hate")\n'
        'parser.add_argument("--only-run", type=int, default=-1, help="[PATCH COLAB] allena solo questo run_id (ripresa run per run)")')
    patch_file(tr,
        "for run_id in range(args.runs):",
        "for run_id in ([args.only_run] if args.only_run >= 0 else range(args.runs)):  # [PATCH COLAB]")
    import difflib
    diff = "".join(difflib.unified_diff(orig.splitlines(True), tr.read_text().splitlines(True), "train_rec.py (autori)", "train_rec.py (colab)"))
    (DIRS["patches"] / "train_rec_only_run.patch").write_text(diff)
    print(diff)
else:
    print("Patch --only-run gia' applicata.")

## Funzioni di supporto
Definizioni usate dalle Parti B, C, D (nessun cambiamento al codice degli autori).

In [ ]:
import argparse, shlex, zlib, pickle
import numpy as np
import pandas as pd
from IPython.display import display, Image

assert int(pd.__version__.split(".")[0]) < 3, "Riavvia la sessione (Runtime > Riavvia sessione) e riesegui dall'inizio: pandas>=3 caricato nel kernel."

PY = sys.executable

def run_logged(cmd, cwd, log_path, env=None, show=r"Epoch\s+(\d+)", show_every=10, tail_on_error=40):
    # Esegue il comando, scrive tutto il log su file (Drive) e stampa solo i progressi (epoca 1 e ogni 10).
    env = {**os.environ, **(env or {})}
    t0 = time.time()
    last = []
    with open(log_path, "a") as lf:
        lf.write(f"\n$ {cmd}\n")
        p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            lf.write(line); last.append(line); last = last[-tail_on_error:]
            m = re.search(show, line) if show else None
            if m and (int(m.group(1)) == 1 or int(m.group(1)) % show_every == 0):
                print(f"   [{(time.time()-t0)/60:6.1f} min] {line.strip()[:150]}")
        rc = p.wait()
    if rc != 0:
        print("".join(last))
        raise RuntimeError(f"Comando fallito (exit {rc}): {cmd}\nLog completo: {log_path}")
    return time.time() - t0

def expected_csv(cmdline):
    # Nome del CSV che ranker_kuairand.py produce (stessa formula/default del codice degli autori, riga 415).
    ap = argparse.ArgumentParser()
    ap.add_argument("--runs", type=int, default=10); ap.add_argument("--epoch", type=int, default=10)
    ap.add_argument("--cores", type=int, default=4); ap.add_argument("--beta", type=float, default=-1.0)
    ap.add_argument("--dataset", default="kuairand"); ap.add_argument("--base-harm", type=float, default=0.3)
    ap.add_argument("--score-model", default="sigformer"); ap.add_argument("--score-type", default="subtraction")
    ap.add_argument("--method", default="remove"); ap.add_argument("--use-single-stage-ranker", action="store_true")
    ap.add_argument("--users", default="all"); ap.add_argument("--accuracy-of-ranker", type=float, default=-1)
    a, _ = ap.parse_known_args(shlex.split(cmdline)[2:])
    return a.score_model, f"results_{a.dataset}_{a.method}_{a.score_model}_{a.score_type}_{a.runs}_{a.beta}_{a.base_harm}_{a.epoch}_{a.use_single_stage_ranker}_{a.users}.csv"

EVAL_SCRIPTS = {   # script ufficiale -> sottocartella (stessa struttura di results/ degli autori)
    "run_experiment_replace_remove.sh": "01_results_remove_replace",
    "run_ablation_beta.sh":             "02_ablation_beta",
    "run_ablation_hard_users.sh":       "03_results_hard_easy_users",
}

def run_eval_scripts(repo, out_root, available_models=None, cores=CORES):
    # Esegue, riga per riga, i comandi di experiments/*.sh (invariati, tranne --cores se diverso da 8).
    # Salta le righe il cui CSV esiste gia' (ripresa). Sposta i CSV prodotti in out_root/<sottocartella>.
    log = DIRS["logs"] / f"eval_{Path(out_root).name}.log"
    for script, sub in EVAL_SCRIPTS.items():
        out = Path(out_root) / sub; out.mkdir(parents=True, exist_ok=True)
        for line in (Path(repo) / "experiments" / script).read_text().splitlines():
            line = line.strip()
            if not line.startswith("python ranker_kuairand.py"):
                continue
            cmd = re.sub(r"--cores\s+\d+", f"--cores {cores}", line)
            model, csv = expected_csv(cmd)
            if (out / csv).exists():
                print(f"[skip] {sub}/{csv} (gia' presente)"); continue
            if available_models is not None and model not in available_models:
                print(f"[manca modello '{model}'] salto: {cmd}"); continue
            print(f"[run ] {sub}: {cmd}")
            dt = run_logged(cmd, cwd=repo, log_path=log, env={"PYTHONPATH": "."}, show=None)
            produced = Path(repo) / csv
            assert produced.exists(), f"CSV atteso non prodotto: {csv}"
            shutil.move(str(produced), str(out / csv))
            print(f"       fatto in {dt/60:.1f} min -> {out/csv}")

KEYS    = ["run_id", "epoch", "base_harm", "beta", "conformal_score", "conformal_method", "alpha", "k", "Method"]
METRICS = ["nDCG @ k", "H(S,X)", "|S|", "random_items", "Recall @ k"]

def compare_exact(ours_root, ref_root):
    # Confronto riga-per-riga con i CSV degli autori (valido quando i punteggi in ingresso sono gli stessi: Parte C).
    rows = []
    for ref in sorted(Path(ref_root).rglob("*.csv")):
        mine = Path(ours_root) / ref.parent.name / ref.name
        if not mine.exists():
            rows.append(dict(file=ref.name, status="NON RIGENERATO (non previsto dagli script ufficiali)")); continue
        a = pd.read_csv(ref, float_precision="high").sort_values(KEYS, kind="stable").reset_index(drop=True)
        b = pd.read_csv(mine, float_precision="high").sort_values(KEYS, kind="stable").reset_index(drop=True)
        if a.shape != b.shape or not a[KEYS].equals(b[KEYS]):
            rows.append(dict(file=ref.name, status=f"STRUTTURA DIVERSA {a.shape} vs {b.shape}")); continue
        d = {m: float((a[m] - b[m]).abs().max()) for m in METRICS if m in a and m in b}
        worst = max(d.values())
        st = "IDENTICO" if worst <= 1e-9 else ("QUASI IDENTICO (<1e-4)" if worst <= 1e-4 else "DIFFERENTE")
        rows.append(dict(file=ref.name, status=st, **{f"maxdiff {m}": v for m, v in d.items()}))
    return pd.DataFrame(rows)

def _add_arel(df):
    # La griglia di alpha dipende dallo split (alpha_max = tasso di danno empirico della run): per allineare le celle fra le 10 run
    # si usa alpha relativo (alpha / alpha_max della run, 3 decimali); alpha = -1 e' la riga "senza controllo" (sentinella).
    amax = df.groupby("run_id")["alpha"].transform("max")
    df = df.copy()
    df["a_rel"] = np.where(df["alpha"] < 0, -1.0, (df["alpha"] / amax.replace(0, np.nan)).round(3))
    return df

def compare_aggregate(ours_root, ref_root):
    # Confronto statistico (Parte D): media e dev.standard sulle 10 run in ogni cella (score, metodo, beta, k, alpha relativo),
    # scarto fra riaddestrato e paper espresso in errori standard (SE).
    G = ["conformal_score", "conformal_method", "beta", "k", "Method", "a_rel"]
    out = []
    for ref in sorted(Path(ref_root).rglob("*.csv")):
        mine = Path(ours_root) / ref.parent.name / ref.name
        if not mine.exists():
            continue
        A = _add_arel(pd.read_csv(ref, float_precision="high")); B = _add_arel(pd.read_csv(mine, float_precision="high"))
        ga = A.groupby(G, dropna=False)[METRICS].agg(["mean", "std", "count"])
        gb = B.groupby(G, dropna=False)[METRICS].agg(["mean", "std", "count"])
        j = pd.concat([ga, gb], axis=1, keys=["ref", "our"], join="inner")
        for m in METRICS:
            mr, mo = j[("ref", m, "mean")], j[("our", m, "mean")]
            se = np.sqrt(j[("ref", m, "std")] ** 2 / j[("ref", m, "count")] + j[("our", m, "std")] ** 2 / j[("our", m, "count")])
            diff = (mo - mr).abs()
            z = np.where(se > 0, diff / se.replace(0, np.nan), np.where(diff == 0, 0.0, np.inf))
            rel = diff / mr.abs().replace(0, np.nan)
            out.append(dict(file=ref.name, metric=m, celle=len(j), media_abs_diff=float(diff.mean()),
                            media_rel_diff_pct=float(100 * rel.mean(skipna=True)),
                            pct_celle_entro_2SE=float(100 * np.mean(np.asarray(z) <= 2)),
                            pct_celle_entro_3SE=float(100 * np.mean(np.asarray(z) <= 3))))
    return pd.DataFrame(out)

def make_figures(repo, csv_root, out_dir):
    # Esegue i 3 script ufficiali di plotting; i PDF finiscono in out_dir (cwd). PYTHONPATH necessario (assente nel README).
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    env = {"PYTHONPATH": str(repo), "MPLBACKEND": "Agg"}
    for script, sub in [("plot_figure_3_and_4.py", "01_results_remove_replace"), ("plot_figure_5.py", "02_ablation_beta"), ("plot_figure_6.py", "03_results_hard_easy_users")]:
        d = Path(csv_root) / sub
        if not d.exists() or not list(d.glob("*.csv")):
            print(f"[salto] {script}: nessun CSV in {d}"); continue
        run_logged(f"{PY} {Path(repo)/'analytics'/script} {d}", cwd=out_dir, log_path=DIRS["logs"] / f"plots_{out_dir.name}.log", env=env, show=None)
    pdfs = sorted(out_dir.glob("*.pdf")); print(f"{len(pdfs)} figure in {out_dir}:", [p.name for p in pdfs])
    return pdfs

def show_pdfs(pdfs, dpi=75):
    if shutil.which("pdftoppm") is None:
        sh("apt-get -qq install -y poppler-utils > /dev/null", check=False)
    for p in pdfs:
        png = Path("/content/_png"); png.mkdir(parents=True, exist_ok=True)
        subprocess.run(["pdftoppm", "-png", "-r", str(dpi), "-singlefile", str(p), str(png / p.stem)], check=False)
        f = png / (p.stem + ".png")
        if f.exists():
            print(p.name); display(Image(filename=str(f)))

_VALID_CACHE = {}
def valid_pickle(path):
    # True se il file esiste ed e' un pickle zlib completo (protegge da file troncati da una disconnessione).
    path = str(path)
    if not os.path.exists(path):
        return False
    key = (path, os.path.getsize(path), os.path.getmtime(path))
    if key not in _VALID_CACHE:
        try:
            with open(path, "rb") as fh:
                obj = pickle.loads(zlib.decompress(fh.read()))
            _VALID_CACHE[key] = isinstance(obj, dict) and len(obj) > 0
        except Exception:
            _VALID_CACHE[key] = False
    return _VALID_CACHE[key]

print("Funzioni definite.")

## Parte B — Figure del paper dai CSV pubblicati dagli autori
Comandi del README (sezione *Replicate our results*), con `PYTHONPATH=.` e `usetex` disattivato.
Verifica che il codice di analisi regga e produce le figure 3–6 del paper partendo dai loro risultati; **non** dimostra ancora nulla sul training.

In [ ]:
def _pdf_to_png(p, dpi):
    png_dir = Path("/content/_png"); png_dir.mkdir(parents=True, exist_ok=True)
    out = png_dir / (Path(p).stem + ".png")
    if shutil.which("pdftoppm"):
        subprocess.run(["pdftoppm", "-png", "-r", str(dpi), "-singlefile", str(p), str(png_dir / Path(p).stem)], check=False)
    if not out.exists():
        try:
            import fitz
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymupdf"], check=False)
            import fitz
        fitz.open(str(p))[0].get_pixmap(dpi=dpi).save(str(out))
    return out

def show_pdfs(pdfs, dpi=75):
    for p in pdfs:
        try:
            f = _pdf_to_png(p, dpi); print(Path(p).name); display(Image(filename=str(f)))
        except Exception as e:
            print(f"[anteprima non disponibile: {e}] -> PDF su Drive: {p}")

## Parte C — Rivalutazione conformal sui punteggi pre-calcolati degli autori
`ranker_kuairand.py` legge (i) i **pickle dei punteggi** (inclusi nel repo) e (ii) gli **split** `methods/kuairand/training/*.txt`.
Gli split **non sono nel repo** (`*.txt` è in `.gitignore`) e il README non lo dice: vanno rigenerati con `train_rec.py --build-only-dataset`
(deterministico: `train_test_split(..., random_state=run_id)`).

Poi si lanciano gli script ufficiali `experiments/*.sh` così come sono (16 comandi × 10 run) e si confrontano i CSV con quelli pubblicati in `results/`.
Se gli ingressi sono gli stessi ci si aspetta risultati **identici**: è una prova di riproducibilità della fase di valutazione.

In [ ]:
# C1) Rigenera gli split (stesso comando e stessi parametri degli autori, flag --build-only-dataset)
sh(f"{PY} train_rec.py --model ncf --epochs {EPOCHS} --runs {RUNS} --build-only-dataset", cwd=REPO_AUTHORS / "methods")
n_split = len(glob.glob(str(REPO_AUTHORS / "methods/kuairand/training/*.txt")))
print("File di split:", n_split, "(attesi 20 = 10 train + 10 test_calibration)")
assert n_split == 20

In [ ]:
# C2) Valutazione conformal (script ufficiali). Riprendibile: i CSV gia' presenti su Drive vengono saltati.
t0 = time.time()
run_eval_scripts(REPO_AUTHORS, DIRS["eval_on_authors"], available_models=None)
print(f"Parte C2 completata in {(time.time()-t0)/60:.1f} min")

In [ ]:
# C3) Confronto numerico con i CSV pubblicati dagli autori
cmp_C = compare_exact(DIRS["eval_on_authors"], REPO_AUTHORS / "results")
pd.set_option("display.width", 250, "display.max_colwidth", 90)
display(cmp_C)
cmp_C.to_csv(D / "confronto_C_valutazione_su_pickle_autori.csv", index=False)
print(cmp_C.status.value_counts().to_string())

In [ ]:
# C4) Figure da CSV rigenerati
pdfs_C = make_figures(REPO_AUTHORS, DIRS["eval_on_authors"], DIRS["figures_eval_on_authors"])
show_pdfs(pdfs_C)

## Parte D — Riaddestramento da zero (100 epoche × 10 run × 6 modelli)

**Attenzione ai tempi.** Non sono stati misurati su una T4: la cella D2 li stima sulla *tua* GPU prima di impegnarti.
Colab gratuito può disconnettersi (limite ~12 h, inattività, quota GPU): per questo ogni run completata è salvata su Drive e **le celle D3/D4 sono rieseguibili** (saltano quanto è già fatto).
Tieni aperta la scheda del browser durante il training.

**Nota sui batch size.** Il README (`gformer 256`, `siren 2048`, `lightgcl 2014` — refuso evidente) e `methods/run_all.sh` (`4096` per tutti e tre) sono incoerenti.
Il notebook usa `BATCH_PROFILE="run_all"` (il README stesso rimanda a `run_all.sh`), ma **non è possibile sapere quale configurazione abbia generato i risultati del paper**:
è una domanda da fare all'autore con cui il professore è in contatto.

In [ ]:
BATCH = {
    "run_all": {"ncf": 8192, "ncfharm": 8192, "sigformer": None, "siren": 4096, "lightgcl": 4096, "gformer": 4096},
    "readme":  {"ncf": 8192, "ncfharm": 8192, "sigformer": None, "siren": 2048, "lightgcl": 2014, "gformer": 256},   # 2014 = valore letterale del README (refuso?)
}[BATCH_PROFILE]

# Ordine: ncf per primo (serve a TUTTE le valutazioni a due stadi), ncfharm per ultimo (non usato da experiments/*.sh)
MODELS_ALL = ["ncf", "lightgcl", "sigformer", "siren", "gformer"] + (["ncfharm"] if INCLUDE_NCFHARM else [])
MODELS_TO_RUN = list(MODELS_ALL)       # <- in una sessione puoi restringerlo, es. ["ncf", "lightgcl"]

TRAIN_DIR = REPO_RETRAIN / "methods"
def final_pkl(model, run): return DIRS["retrain_pickles"] / f"test_{run}_{model}_False_0.3_{EPOCHS}.zlib.pickle"
def train_cmd(model, run_arg):
    c = f"{PY} train_rec.py --model {model} --epochs {EPOCHS} --runs {RUNS} --gpu {GPU_INDEX}"
    if BATCH[model]: c += f" --batch-size {BATCH[model]}"
    return c + (f" --only-run {run_arg}" if run_arg is not None else "")

def status_table():
    rows = []
    for m in MODELS_ALL:
        done = [r for r in range(RUNS) if final_pkl(m, r).exists() and valid_pickle(final_pkl(m, r))]
        rows.append(dict(modello=m, run_complete=f"{len(done)}/{RUNS}", mancanti=[r for r in range(RUNS) if r not in done]))
    return pd.DataFrame(rows)

# D1) Rigenera gli split anche nella copia retrain e verifica che siano identici byte-per-byte a quelli della copia 'authors'
sh(f"{PY} train_rec.py --model ncf --epochs {EPOCHS} --runs {RUNS} --build-only-dataset", cwd=TRAIN_DIR)
import hashlib
def md5(p): return hashlib.md5(open(p, "rb").read()).hexdigest()
same = all(md5(a) == md5(REPO_RETRAIN / "methods/kuairand/training" / Path(a).name)
           for a in glob.glob(str(REPO_AUTHORS / "methods/kuairand/training/*.txt")))
print("Split identici tra le due copie:", same)
assert same
display(status_table())

In [ ]:
# D2) (Facoltativo ma consigliato) Misura di tempo sulla TUA GPU: 1 e 3 epoche, run 0, per ogni modello -> stima del tempo totale.
# Non scrive pickle (vengono salvati solo alle epoche multiple di 10).
RUN_TIMING_PROBE = True
if RUN_TIMING_PROBE:
    est = []
    for m in MODELS_TO_RUN:
        ts = {}
        for ep in (1, 3):
            cmd = train_cmd(m, 0).replace(f"--epochs {EPOCHS}", f"--epochs {ep}")
            ts[ep] = run_logged(cmd, cwd=TRAIN_DIR, log_path=DIRS["logs"] / "timing_probe.log", show=None)
        per_epoch = max((ts[3] - ts[1]) / 2, 0.0); fixed = max(ts[1] - per_epoch, 0.0)
        run_min = (fixed + EPOCHS * per_epoch) / 60
        est.append(dict(modello=m, s_per_epoca=round(per_epoch, 1), overhead_s=round(fixed), min_per_run=round(run_min, 1), ore_10_run=round(RUNS * run_min / 60, 1)))
        print(est[-1])
    est = pd.DataFrame(est); display(est)
    print(f"STIMA TOTALE ~{est.ore_10_run.sum():.1f} h di GPU (esclude le 10 valutazioni intermedie per run: potrebbero aggiungere molto). Sessioni Colab da ~{SESSION_BUDGET_HOURS} h necessarie: ~{est.ore_10_run.sum()/SESSION_BUDGET_HOURS:.1f}")

In [ ]:
# D3) Training con ripresa. Rieseguire questa cella dopo ogni disconnessione.
def train_all():
    t_start = time.time(); longest = {}
    for m in MODELS_TO_RUN:
        if not PER_RUN_RESUME:
            # Comando ESATTO degli autori: un solo processo per modello, tutte e 10 le run (nessuna ripresa a meta').
            if all(valid_pickle(final_pkl(m, r)) for r in range(RUNS)):
                print(f"[ok  ] {m}: gia' completo"); continue
            print(f"[train] {m} (comando degli autori, {RUNS} run in un processo)")
            dt = run_logged(train_cmd(m, None), cwd=TRAIN_DIR, log_path=DIRS["logs"] / f"train_{m}.log")
            print(f"        {m} completato in {dt/3600:.2f} h")
        else:
            for r in range(RUNS):
                if valid_pickle(final_pkl(m, r)):
                    continue
                elapsed_h = (time.time() - t_start) / 3600
                if elapsed_h + longest.get(m, 0) / 3600 > SESSION_BUDGET_HOURS:
                    print(f"Budget di sessione ({SESSION_BUDGET_HOURS} h) quasi esaurito: mi fermo. Riesegui la cella in una nuova sessione."); return
                print(f"[train] {m} run {r}  (sessione: {elapsed_h:.2f} h)")
                dt = run_logged(train_cmd(m, r), cwd=TRAIN_DIR, log_path=DIRS["logs"] / f"train_{m}.log")
                longest[m] = max(longest.get(m, 0), dt)
                assert valid_pickle(final_pkl(m, r)), f"Pickle finale non valido per {m} run {r}"
                if not KEEP_INTERMEDIATE_CKPT:
                    for f in glob.glob(str(DIRS["retrain_pickles"] / f"test_{r}_{m}_False_0.3_*.zlib.pickle")):
                        if not f.endswith(f"_{EPOCHS}.zlib.pickle"):
                            os.remove(f)
                print(f"        run {r} completata in {dt/60:.1f} min")
    print("Tutti i modelli selezionati sono completi.")

train_all()
display(status_table())

In [ ]:
# D4) Valutazione conformal sui pickle RIADDESTRATI (script ufficiali; vengono eseguite solo le righe i cui modelli sono completi)
st = status_table()
complete = set(st[st.run_complete == f"{RUNS}/{RUNS}"].modello)
print("Modelli completi:", sorted(complete))
if "ncf" not in complete:
    print("ATTENZIONE: serve ncf completo (valutazione a due stadi). Completa D3 per ncf prima.")
else:
    run_eval_scripts(REPO_RETRAIN, DIRS["eval_on_retrained"], available_models=complete)

In [ ]:
# D5) Confronto con i CSV pubblicati: medie sulle 10 run e scarto in unita' di errore standard.
# Non e' atteso un risultato bit-identico (training su GPU non deterministico, seed per-run diverso se PER_RUN_RESUME=True).
cmp_D = compare_aggregate(DIRS["eval_on_retrained"], REPO_AUTHORS / "results")
display(cmp_D)
cmp_D.to_csv(D / "confronto_D_riaddestramento_vs_paper.csv", index=False)
if len(cmp_D):
    print(cmp_D.groupby("metric")[["media_rel_diff_pct", "pct_celle_entro_2SE", "pct_celle_entro_3SE"]].mean().round(2).to_string())

pdfs_D = make_figures(REPO_RETRAIN, DIRS["eval_on_retrained"], DIRS["figures_retrained"])
show_pdfs(pdfs_D)

## Riepilogo e cosa portare al professore

In [ ]:
print("== Stato Drive:", D)
for k, v in DIRS.items():
    n = sum(1 for _ in v.rglob("*") if _.is_file())
    print(f"  {k:26s} {n:4d} file")
print()
display(status_table())
print("\nParte C (valutazione su pickle degli autori):")
print(cmp_C.status.value_counts().to_string() if "cmp_C" in globals() else "  non ancora eseguita")

### Osservazioni di riproducibilità emerse (da riportare nel report / all'autore)
1. `requirement.txt` (nome al singolare) **non elenca `torchsparsegradutils`**, importata da `methods/SIGformer/model.py`; poiché `train_rec.py` importa tutti i modelli all'avvio, senza di essa nessun training parte.
2. Gli **split** (`methods/kuairand/training/*.txt`) non sono nel repo (`.gitignore: *.txt`), ma `ranker_kuairand.py` li richiede: la sola sezione *Run again all the evaluations* del README non basta; serve `train_rec.py --build-only-dataset`.
3. I comandi di plotting del README non impostano `PYTHONPATH`, ma `plot_figure_5.py` e `plot_figure_6.py` fanno `from analytics...`.
4. Gli script usano `include_groups=True` in `groupby.apply`: **incompatibile con pandas ≥ 3**.
5. Incoerenza fra README e `methods/run_all.sh` sui batch size (e `lightgcl 2014` nel README); `ncfharm` è in `run_all.sh` ma non nel README né usato da `experiments/*.sh`.
6. Gli script `experiments/run_ablation_beta.sh` rigenerano 4 valori di β (−1, 0, 0.5, 1) mentre `results/02_ablation_beta/` ne contiene 6 (anche 0.1 e 1.5): quei due CSV non sono riproducibili con gli script forniti.